# 🎯 Smart Resume Keyword Matcher

> **Portfolio Project** | Classical NLP + TF-IDF + Cosine Similarity
>
> Compares a Resume PDF against a Job Description and returns:
> - An overall **match score (%)**
> - A **section breakdown** (Skills / Experience / Education)
> - A prioritized list of **missing keywords**
> - A **history log** of all past comparisons

---

**Tech Stack:** `pdfplumber` · `spaCy (en_core_web_sm)` · `scikit-learn (TF-IDF)` · `Gradio`

**Setup:**
```bash
pip install -r requirements.txt
python -m spacy download en_core_web_sm
```

## ⚙️ Setup & Imports

In [ ]:
# ── Setup Verification ────────────────────────────────────────────────────────
# Run this cell FIRST to verify all dependencies are installed correctly.
# If any import fails, run: pip install -r requirements.txt
# Then run: python -m spacy download en_core_web_sm

import sys
import importlib
import os

REQUIRED = ["pdfplumber", "spacy", "sklearn", "gradio", "pandas", "numpy"]
missing = [pkg for pkg in REQUIRED if importlib.util.find_spec(pkg) is None]

if missing:
    print(f"❌ Missing packages: {missing}")
    print("   Fix: pip install -r requirements.txt")
else:
    print("✅ All dependencies found. You're offline-ready.")

# Load spaCy model — works fully offline after the first download
import spacy
try:
    nlp = spacy.load("en_core_web_sm")
    print("✅ spaCy model 'en_core_web_sm' loaded.")
except OSError:
    print("❌ spaCy model not found.")
    print("   Fix: python -m spacy download en_core_web_sm")

# Verify history.json exists — create it if missing
HISTORY_PATH_INIT = "history.json"
if not os.path.exists(HISTORY_PATH_INIT):
    import json
    with open(HISTORY_PATH_INIT, "w") as f:
        json.dump([], f)
    print(f"✅ Created {HISTORY_PATH_INIT}")
else:
    print(f"✅ {HISTORY_PATH_INIT} found.")

print("\n🚀 Setup complete — ready to match resumes!")

## 📄 Phase 1: PDF Parsing & Section Detection

These two functions form the **input pipeline**:
1. `extract_text_from_pdf()` — reads a PDF and returns raw text
2. `detect_sections()` — splits resume text into Skills / Experience / Education buckets

In [ ]:
# ── PDF Text Extractor ────────────────────────────────────────────────────────
# pdfplumber handles multi-column layouts better than PyPDF2 — important for
# modern resume templates that use a two-column format.

import pdfplumber
import re

def extract_text_from_pdf(pdf_path: str) -> str:
    """
    Extract all text from a PDF file, page by page.

    Returns cleaned text string, or empty string on failure.
    """
    full_text = []
    try:
        with pdfplumber.open(pdf_path) as pdf:
            for page_num, page in enumerate(pdf.pages, start=1):
                page_text = page.extract_text()
                if page_text:
                    full_text.append(page_text)
                else:
                    print(f"  ⚠️ Page {page_num}: No text (may be image-based)")
    except FileNotFoundError:
        print(f"❌ File not found: {pdf_path}")
        return ""
    except Exception as e:
        print(f"❌ Error reading PDF: {e}")
        return ""

    combined = "\n".join(full_text)
    combined = re.sub(r'\n{3,}', '\n\n', combined)
    combined = re.sub(r'[ \t]{2,}', ' ', combined)
    return combined.strip()


# Smoke Test: uncomment to test
# sample_text = extract_text_from_pdf("sample_resumes/test_resume.pdf")
# print(sample_text[:500])

In [ ]:
# ── Section Detector ─────────────────────────────────────────────────────────
# Heuristically splits resume text into Skills, Experience, Education, General.
# The 40-char guard prevents long sentences from being misread as section headers.

def detect_sections(text: str) -> dict:
    """
    Split resume text into section buckets using keyword detection.

    Returns dict with keys: skills, experience, education, general.
    """
    if not text:
        return {"skills": "", "experience": "", "education": "", "general": ""}

    SECTION_KEYWORDS = {
        "skills":     ["skill", "technical skill", "core competenc", "technologies",
                       "tools", "programming", "languages", "proficienc"],
        "experience": ["experience", "work history", "employment", "career",
                       "professional background", "work experience", "internship"],
        "education":  ["education", "academic", "qualification", "degree",
                       "university", "college", "schooling", "certification"],
    }

    sections = {"skills": [], "experience": [], "education": [], "general": []}
    current_section = "general"

    for line in text.split("\n"):
        line_lower = line.lower().strip()
        matched_section = None
        if len(line_lower) <= 40:
            for section, keywords in SECTION_KEYWORDS.items():
                if any(kw in line_lower for kw in keywords):
                    matched_section = section
                    break
        if matched_section:
            current_section = matched_section
        else:
            sections[current_section].append(line)

    return {k: "\n".join(v).strip() for k, v in sections.items()}


# Smoke Test: uncomment to test
# sample_sections = detect_sections(sample_text)
# for s, c in sample_sections.items(): print(f"\n── {s.upper()} ──\n{c[:150]}")

## 🧠 Phase 2: NLP Scoring

Four functions forming the **scoring and analysis pipeline**:
1. `preprocess_text()` — lemmatize and clean text for TF-IDF
2. `compute_match_score()` — TF-IDF + cosine similarity → 4 scores
3. `find_missing_keywords()` — JD terms absent from resume, ranked
4. `format_score_output()` — data contract for Phase 3 & 4

In [ ]:
# ── Text Preprocessor ────────────────────────────────────────────────────────
# spaCy lemmatizer reduces words to root forms so "managing" and "managed"
# are treated as the same signal — critical for semantic matching.

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def preprocess_text(text: str) -> str:
    """
    Lowercase, remove stopwords, and lemmatize text for TF-IDF input.

    Returns space-joined token string, or empty string on empty input.
    """
    if not text or not text.strip():
        return ""

    # Disable parser and NER — only need tokenizer + lemmatizer for speed
    doc = nlp(text.lower(), disable=["parser", "ner"])

    clean_tokens = [
        token.lemma_
        for token in doc
        if token.is_alpha
        and not token.is_stop
        and len(token.lemma_) > 2
    ]

    return " ".join(clean_tokens)


# Smoke Test:
# print(preprocess_text("Managing 5 databases and developing Python APIs for the team."))

In [ ]:
# ── Match Scorer ──────────────────────────────────────────────────────────────
# Computes 4 cosine similarity scores: overall + 3 section-level.
# A fresh TfidfVectorizer is created per pair — no state leak between calls.

def compute_match_score(resume_text: str, jd_text: str,
                         resume_sections: dict) -> dict:
    """
    Compute TF-IDF cosine similarity scores between resume and JD.

    Returns dict: {overall, skills, experience, education} — floats 0-100.
    """
    def _score_pair(text_a: str, text_b: str) -> float:
        clean_a, clean_b = preprocess_text(text_a), preprocess_text(text_b)
        if not clean_a or not clean_b:
            return 0.0
        vectorizer = TfidfVectorizer()
        try:
            tfidf_matrix = vectorizer.fit_transform([clean_a, clean_b])
        except ValueError:
            return 0.0
        score = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])[0][0]
        return round(float(score) * 100, 1)

    return {
        "overall":    _score_pair(resume_text, jd_text),
        "skills":     _score_pair(resume_sections.get("skills",     ""), jd_text),
        "experience": _score_pair(resume_sections.get("experience", ""), jd_text),
        "education":  _score_pair(resume_sections.get("education",  ""), jd_text),
    }


# Smoke Test:
# resume = "Python developer with 3 years of machine learning and SQL."
# jd     = "Looking for a Python engineer skilled in data science and SQL."
# print(compute_match_score(resume, jd, detect_sections(resume)))

In [ ]:
# ── Missing Keyword Extractor ─────────────────────────────────────────────────
# Ranks JD keywords by TF-IDF importance, then filters to those absent from resume.
# Smarter than simple set-difference: common words naturally score lower.

def find_missing_keywords(resume_text: str, jd_text: str,
                            top_n: int = 15) -> list:
    """
    Find the most important JD keywords missing from the resume.

    Returns list of strings ranked by TF-IDF weight (highest first).
    """
    if not resume_text or not jd_text:
        return []

    clean_resume = preprocess_text(resume_text)
    clean_jd     = preprocess_text(jd_text)
    if not clean_jd:
        return []

    resume_tokens = set(clean_resume.split())

    vectorizer = TfidfVectorizer()
    try:
        vectorizer.fit([clean_jd])
    except ValueError:
        return []

    feature_names = vectorizer.get_feature_names_out()
    tfidf_scores  = vectorizer.transform([clean_jd]).toarray()[0]
    jd_terms_ranked = sorted(zip(feature_names, tfidf_scores),
                             key=lambda x: x[1], reverse=True)

    missing = [term for term, _ in jd_terms_ranked if term not in resume_tokens]
    return missing[:top_n]


# Smoke Test:
# print(find_missing_keywords(
#     "Python developer with SQL experience.",
#     "Seeking machine learning engineer with Kubernetes and TensorFlow.",
#     top_n=10
# ))

In [ ]:
# ── Score Output Formatter ────────────────────────────────────────────────────
# DATA CONTRACT — both Phase 3 (history) and Phase 4 (Gradio UI) consume this dict.
# Change a key here → update Phase 3 and Phase 4 consumers accordingly.

from datetime import datetime

def format_score_output(resume_filename: str, jd_snippet: str,
                         scores: dict, missing_keywords: list) -> dict:
    """
    Package NLP results into a standardized 8-key output dict.

    Keys: timestamp, resume_filename, jd_snippet, overall_score,
          skills_score, experience_score, education_score, missing_keywords.
    """
    return {
        "timestamp":        datetime.now().isoformat(timespec="seconds"),
        "resume_filename":  resume_filename,
        "jd_snippet":       jd_snippet[:200].strip(),
        "overall_score":    scores.get("overall",    0.0),
        "skills_score":     scores.get("skills",     0.0),
        "experience_score": scores.get("experience", 0.0),
        "education_score":  scores.get("education",  0.0),
        "missing_keywords": missing_keywords,
    }


# Smoke Test:
# result = format_score_output(
#     "my_resume.pdf",
#     "Looking for a Python engineer...",
#     {"overall": 72.5, "skills": 65.0, "experience": 80.0, "education": 45.0},
#     ["kubernetes", "docker"]
# )
# import json; print(json.dumps(result, indent=2))

## 💾 Phase 3: History & Persistence

Three functions for **local JSON history**:
1. `save_result()` — append a result dict to `history.json`
2. `load_history()` — read all past comparison results
3. `history_to_dataframe()` — convert history to a Gradio-ready DataFrame

In [ ]:
# ── History Persistence ───────────────────────────────────────────────────────
# Flat JSON file — simple, portable, zero extra dependencies.
# Design: always a valid JSON array. Read-modify-write on every save.
# save_result() never crashes — history failure should never block the UI.

import json
from pathlib import Path

HISTORY_PATH = Path("history.json")

def save_result(result: dict) -> bool:
    """
    Append a format_score_output() dict to history.json.

    Uses read-modify-write to preserve all previous entries.
    Returns True on success, False on error (never raises).
    """
    try:
        history = []
        if HISTORY_PATH.exists():
            with open(HISTORY_PATH, "r", encoding="utf-8") as f:
                history = json.load(f)

        history.append(result)

        with open(HISTORY_PATH, "w", encoding="utf-8") as f:
            json.dump(history, f, indent=2, ensure_ascii=False)

        return True

    except (OSError, json.JSONDecodeError) as e:
        # Non-critical failure — log it but don't interrupt the UI
        print(f"⚠️ Could not save to history: {e}")
        return False


def load_history() -> list:
    """
    Load all past results from history.json.

    Returns list of dicts (oldest first), or [] on any error.
    """
    if not HISTORY_PATH.exists():
        return []
    try:
        with open(HISTORY_PATH, "r", encoding="utf-8") as f:
            history = json.load(f)
        if not isinstance(history, list):
            print("⚠️ history.json is malformed — returning empty history.")
            return []
        return history
    except (OSError, json.JSONDecodeError) as e:
        print(f"⚠️ Could not read history: {e}")
        return []


# Smoke Tests:
# test_entry = format_score_output(
#     "test.pdf", "Seeking a Python engineer...",
#     {"overall": 72.5, "skills": 65.0, "experience": 80.0, "education": 45.0},
#     ["kubernetes", "docker"]
# )
# print("Saved:", save_result(test_entry))
#
# Test load:
# history = load_history()
# print(f"History has {len(history)} entries.")

In [ ]:
# ── History Display Helper ────────────────────────────────────────────────────
# Converts raw history into a pandas DataFrame ready for Gradio's gr.Dataframe.
# Lives in Phase 3 (next to the data it transforms) rather than Phase 4 (UI).
# Keeping transformation separate from display makes both easier to test.

import pandas as pd

# Column order and names optimized for the Gradio History tab
_HISTORY_COLUMNS = [
    "Time", "Resume", "Overall %", "Skills %",
    "Experience %", "Education %", "Missing Keywords"
]

def history_to_dataframe() -> pd.DataFrame:
    """
    Load history and return it as a formatted DataFrame for Gradio display.

    Returns:
        DataFrame with 7 columns, most recent result first.
        Returns empty DataFrame (correct columns) if no history exists.
    """
    history = load_history()

    # Always return a properly-structured DataFrame — never None, never []
    if not history:
        return pd.DataFrame(columns=_HISTORY_COLUMNS)

    rows = [
        {
            "Time":             entry.get("timestamp",        ""),
            "Resume":           entry.get("resume_filename",  ""),
            "Overall %":        entry.get("overall_score",    0.0),
            "Skills %":         entry.get("skills_score",     0.0),
            "Experience %":     entry.get("experience_score", 0.0),
            "Education %":      entry.get("education_score",  0.0),
            "Missing Keywords": ", ".join(entry.get("missing_keywords", [])),
        }
        for entry in history
    ]

    df = pd.DataFrame(rows, columns=_HISTORY_COLUMNS)
    # Most recent first — more useful in the UI
    return df.iloc[::-1].reset_index(drop=True)


# Smoke Test:
# df = history_to_dataframe()
# print(df.shape, df.columns.tolist())
# display(df)   # Works in Jupyter

## 🖥️ Phase 4: Gradio UI

The final piece: the **web interface** that connects everything.
1. `analyze_resume_flow()` — Orchestrator connecting Phase 1-3
2. `gr.Blocks` Layout — Premium, multi-tabbed interface

In [ ]:
# ── Analyze Orchestrator ──────────────────────────────────────────────────────
# This is the 'glue' function that runs the entire pipeline when the User
# clicks 'Analyze'. It connects Phase 1 (Parsing), Phase 2 (NLP), and 
# Phase 3 (History).

def analyze_resume_flow(pdf_file, jd_text: str):
    """
    Orchestrate the full resume matching pipeline.
    
    Args:
        pdf_file: Gradio file object (comes from gr.File).
        jd_text: Raw string from the Job Description text area.
        
    Returns:
        A tuple of (Overall Score %, Section Table Data, Missing Keywords List)
    """
    if not pdf_file or not jd_text.strip():
        return "0%", [], "Please upload a resume and provide a job description."

    # Phase 1: Extract and Detect
    resume_text = extract_text_from_pdf(pdf_file.name)
    if not resume_text:
        return "0%", [], "Failed to extract text from PDF. Is it a scanned image?"
        
    sections = detect_sections(resume_text)
    
    # Phase 2: Score and Analysis
    scores = compute_match_score(resume_text, jd_text, sections)
    missing_kw = find_missing_keywords(resume_text, jd_text)
    
    # Phase 3: Format and Save
    result_dict = format_score_output(
        resume_filename=os.path.basename(pdf_file.name),
        jd_snippet=jd_text,
        scores=scores,
        missing_keywords=missing_kw
    )
    save_result(result_dict)
    
    # Format for UI display
    overall_display = f"{scores['overall']}%"
    
    # Prepare section breakdown table for gr.Dataframe
    section_table = [
        ["Skills Match", f"{scores['skills']}%"],
        ["Experience Match", f"{scores['experience']}%"],
        ["Education Match", f"{scores['education']}%"]
    ]
    
    missing_display = ", ".join(missing_kw) if missing_kw else "None found! Good match."
    
    return overall_display, section_table, missing_display

In [ ]:
# ── Gradio Web Interface ──────────────────────────────────────────────────────
# A premium, two-tabbed interface for analysis and history research.

import gradio as gr

# Modern CSS for a clean look
custom_css = """
.gradio-container { font-family: 'Inter', sans-serif; }
.main-header { text-align: center; color: #2D3748; margin-bottom: 2rem; }
.score-box { background: #F7FAFC; border-radius: 8px; padding: 1rem; border: 1px solid #E2E8F0; }
"""

with gr.Blocks(css=custom_css, title="Smart Resume Matcher") as demo:
    gr.Markdown("# 🎯 Smart Resume Keyword Matcher", elem_classes="main-header")
    
    with gr.Tabs():
        with gr.TabItem("🔍 Analysis"):
            with gr.Row():
                with gr.Column(scale=1):
                    resume_input = gr.File(label="Upload Resume (PDF)", file_types=[".pdf"])
                    jd_input = gr.Textbox(label="Job Description", placeholder="Paste job description here...", lines=15)
                    analyze_btn = gr.Button("🚀 Analyze Match", variant="primary")
                    
                with gr.Column(scale=1):
                    with gr.Group(elem_classes="score-box"):
                        overall_out = gr.Label(label="Overall Match Score")
                        section_out = gr.Dataframe(
                            headers=["Section", "Score"],
                            datatype=["str", "str"],
                            label="Section Breakdown"
                        )
                    missing_out = gr.Textbox(label="Missing Keywords (Ranked by Importance)", lines=5)
        
        with gr.TabItem("📜 History"):
            history_table = gr.Dataframe(
                value=history_to_dataframe(),
                label="Comparison History (Most Recent First)",
                interactive=False
            )
            refresh_btn = gr.Button("🔄 Refresh History")

    # Event: Run Analysis
    analyze_btn.click(
        fn=analyze_resume_flow,
        inputs=[resume_input, jd_input],
        outputs=[overall_out, section_out, missing_out]
    )
    
    # Event: Refresh History
    refresh_btn.click(
        fn=history_to_dataframe,
        outputs=history_table
    )

print("✅ Gradio UI code ready. Run demo.launch() to start!")